# mT5-small Kaggle 训练版

这份 notebook 目标是尽量做到按顺序执行即可完成：安装依赖、加载数据、构建数据集、微调 mT5-small、保存模型。

In [ ]:
# 这一格建议先运行
# 在 notebook 里优先使用 %pip，而不是 !pip

%pip install -q transformers datasets sentencepiece accelerate

In [ ]:
# 导入依赖
import json
import os
from pathlib import Path

import torch
import transformers
import datasets
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# 配置模型和路径
# 如果你把数据上传到 Kaggle Dataset，优先改这里的候选路径

MODEL_NAME = "google/mt5-small"
OUTPUT_DIR = Path("./mt5_eda_output")

DATA_CANDIDATES = [
    Path("./datasets/eda_train_50000.jsonl"),
    Path("/kaggle/input/eda-generate/eda_train_50000.jsonl"),
    Path("/kaggle/input/eda-generate/datasets/eda_train_50000.jsonl"),
]

data_path = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError("没有找到 eda_train_50000.jsonl，请先确认 Kaggle 数据集挂载路径。")

print("model:", MODEL_NAME)
print("data path:", data_path)
print("output dir:", OUTPUT_DIR)

In [ ]:
# 训练超参数
# 第一版先按 Kaggle T4 / P100 能接受的保守配置写

MAX_INPUT_LENGTH = 192
MAX_TARGET_LENGTH = 512
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACC_STEPS = 8
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
VAL_RATIO = 0.05

# 如果只是先验证流程，可以把 SAMPLE_LIMIT 改成 1000 或 5000
SAMPLE_LIMIT = None

In [ ]:
# 读取 JSONL 数据
# 这里把 output 压成紧凑 JSON 字符串，方便模型学习稳定格式

records = []
with data_path.open("r", encoding="utf-8") as fh:
    for idx, line in enumerate(fh):
        if not line.strip():
            continue
        item = json.loads(line)
        records.append({
            "input_text": item["prompt"],
            "target_text": json.dumps(item["output"], ensure_ascii=False, separators=(",", ":"))
        })
        if SAMPLE_LIMIT is not None and len(records) >= SAMPLE_LIMIT:
            break

print("loaded samples:", len(records))
print("input example:")
print(records[0]["input_text"])
print()
print("target example:")
print(records[0]["target_text"][:600])

In [ ]:
# 构建 Dataset 并切分训练集 / 验证集

dataset = Dataset.from_list(records)
split_dataset = dataset.train_test_split(test_size=VAL_RATIO, seed=42)

dataset_dict = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"]
})

print(dataset_dict)

In [ ]:
# 加载 tokenizer 和模型
# mT5 这里明确使用 use_fast=False，避免 sentencepiece 相关问题

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# 这一项可以略微省显存
model.gradient_checkpointing_enable()

print("tokenizer and model ready")

In [ ]:
# 定义 tokenize 逻辑

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# 进行 tokenize

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
    desc="tokenizing dataset",
)

print(tokenized_datasets)

In [ ]:
# 数据整理器

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("data collator ready")

In [ ]:
# 训练参数
# 这里尽量避免太激进，先保证 Kaggle 上能跑通

use_fp16 = torch.cuda.is_available()

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=use_fp16,
    report_to="none",
    load_best_model_at_end=False,
)

print(training_args)

In [ ]:
# 构建 trainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("trainer ready")

In [ ]:
# 开始训练
# 如果只是做结构检查，可以先不跑这一格

train_result = trainer.train()
train_result

In [ ]:
# 保存模型和 tokenizer

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("saved to:", OUTPUT_DIR)

In [ ]:
# 简单推理测试

test_prompt = "设计一个完成RC低通滤波功能的 EDA 原理图，包含输入 VIN、电阻 R1、输出 OUT、电容 C1 和 接地 GND。请给出所有节点的坐标以及节点之间的连接关系。"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH)
inputs = {k: v.to(device) for k, v in inputs.items()}

generated_ids = model.generate(
    **inputs,
    max_length=MAX_TARGET_LENGTH,
)

result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(result)